# Discriminant Classifiers

Using discriminant classifiers / Naive Bayes to classify our data.

In [1]:
import pandas as pd

In [2]:
learn_data = pd.read_csv("resampled_train_v2.csv", header = None)
learn_data.columns = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,AR,BilRatio,Female,Target
0,0.173844,1.109406,1.197304,0.362037,-1.391442,0.487504,0.529868,-0.903219,-1.399329,1.203258,0,0
1,-0.379308,0.226878,0.469206,-0.573270,0.069730,0.288250,0.908406,1.482342,1.488320,0.927290,0,0
2,-1.362690,-0.430090,-0.383317,-0.232378,0.039705,0.575302,-0.227207,-0.024328,0.212382,-0.353374,0,0
3,-0.194924,-0.795164,-0.697958,-0.925510,-0.157437,0.589292,-0.227207,0.101228,0.413846,-0.458710,1,0
4,0.542612,2.761282,2.439451,1.783803,-0.349504,-0.293097,1.286944,0.352339,-0.459164,1.153956,1,0


In [3]:
learn_data.isna().value_counts()

Age    TB     DB     Alkphos  Sgpt   Sgot   TP     ALB    AR     BilRatio  Female  Target
False  False  False  False    False  False  False  False  False  False     False   False     549
Name: count, dtype: int64

In [4]:
X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

## Metrics

In [5]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall_0, recall_1, prec_0, prec_1, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear Discriminant Classifier

From PCA analysis, we know that our data can be separated in two clouds of sick and healthy patients respectively. A LD classifier might work well, but we know that the clouds may overlap. Also, their covariance matrices are clearly different. We might need to use a Quadratic Discriminant Classifier instead.

In [7]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.33, random_state = 42)

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
lda_model.fit(X_train, y_train)

print('Priors:', lda_model.priors_)

Priors: [0.5 0.5]


In [8]:
confusion(np.array(y_train), pd.Series(lda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	124	25
	0	92	126
Accuracy: 68.12%


In [9]:
confusion(np.array(y_val), pd.Series(lda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	63	14
	0	42	63
Accuracy: 69.23%


In [11]:
from sklearn.model_selection import cross_validate

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
cross_val_results = pd.DataFrame(cross_validate(lda_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["LDA", :] = mean_results
metrics_df

,F1 Macro,Recall,Precision,Accuracy
LDA,0.682904,0.703997,0.704123,0.683169


## Quadratic Discrimant Classifier

We now use a QDA classifier. We see that the problem is preserved: the "sick" class overlaps too much with the healthy class and it

In [12]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
_ = qda_model.fit(X_train, y_train)

In [13]:
confusion(np.array(y_train), pd.Series(qda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	133	16
	0	91	127
Accuracy: 70.84%


In [14]:
confusion(np.array(y_val), pd.Series(qda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	68	9
	0	43	62
Accuracy: 71.43%


In [15]:
qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
cross_val_results = pd.DataFrame(cross_validate(qda_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["QDA", :] = mean_results
metrics_df

,F1 Macro,Recall,Precision,Accuracy
LDA,0.682904,0.703997,0.704123,0.683169
QDA,0.706038,0.731229,0.736336,0.706789


QDA can be regularized with a parameter between 0 and 1, so we can apply cross-validation in an attempt to obtain better metrics. In general, a small value of this regularization parameter (between 0.01 and 0.1) is desirable, but it does not improve results by much.

In [16]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

regs = np.logspace(start = -4, stop = -0.5, num = 100)

for reg in regs:
    qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5), reg_param = reg)
    this_results = pd.DataFrame(cross_validate(qda_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[reg, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
0.000100,0.707852,0.732768,0.737588,0.708607
0.000108,0.707852,0.732768,0.737588,0.708607
0.000192,0.706038,0.731229,0.736336,0.706789
0.000313,0.706038,0.731229,0.736336,0.706789
0.000266,0.706038,0.731229,0.736336,0.706789


## Naive Bayes



In [17]:
from sklearn.naive_bayes import GaussianNB

gaussian_nb = GaussianNB(priors = (0.5, 0.5))
gaussian_nb.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(gaussian_nb.predict(X_train)))

		Predicted
		+1	0
Real	+1	130	19
	0	96	122
Accuracy: 68.66%


In [18]:
confusion(np.array(y_val), pd.Series(gaussian_nb.predict(X_val)))

		Predicted
		+1	0
Real	+1	66	11
	0	44	61
Accuracy: 69.78%


In [19]:
gaussian_nb = GaussianNB(priors = (0.5, 0.5))
cross_val_results = pd.DataFrame(cross_validate(gaussian_nb, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["GaussianNB", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.706038,0.731229,0.736336,0.706789
GaussianNB,0.689961,0.716796,0.722233,0.690359
LDA,0.682904,0.703997,0.704123,0.683169


## Logistic Regression

In [24]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

logreg_model = LogisticRegression(C = 20, random_state = 42, class_weight = "balanced")

logreg_model.fit(X_train, y_train)
confusion(np.array(y_train), pd.Series(logreg_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	121	28
	0	79	139
Accuracy: 70.84%


In [25]:
confusion(np.array(y_val), pd.Series(logreg_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	61	16
	0	37	68
Accuracy: 70.88%


In [26]:
Cs = np.logspace(start = -4, stop = 2, num = 200)

logreg_model = LogisticRegressionCV(Cs = Cs, random_state = 42, cv = 5, class_weight = "balanced")

logreg_model.fit(X, y)
confusion(np.array(y), pd.Series(logreg_model.predict(X)))

		Predicted
		+1	0
Real	+1	196	30
	0	126	197
Accuracy: 71.58%


In [27]:
avg_crossval_scores = logreg_model.scores_[1].mean(axis=0)
idx = np.argmax(avg_crossval_scores)
best_C = logreg_model.Cs_[idx]
print(best_C)

0.017027691722258993


In [28]:
logreg_model_best = LogisticRegression(C = best_C, random_state = 42)
cross_val_results = pd.DataFrame(cross_validate(logreg_model_best, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.706038,0.731229,0.736336,0.706789
GaussianNB,0.689961,0.716796,0.722233,0.690359
LogReg-Best,0.689349,0.692313,0.689434,0.695947
LDA,0.682904,0.703997,0.704123,0.683169


## Trying our best classifiers on our test data

In [29]:
test_data = pd.read_csv("scaled_test.csv", header = None)
test_data.columns = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,AR,BilRatio,Female
0,-2.100226,-0.795164,-1.235840,1.907027,-0.527803,-0.567456,0.624503,1.356786,1.555475,-1.512071,0
1,1.034303,0.171538,0.469206,-0.117670,0.688275,1.320148,2.044019,1.105674,-0.459164,1.121330,0
2,0.911380,-0.795164,-0.697958,-0.643898,-0.269091,-1.387576,1.286944,1.356786,0.548155,-0.458710,0
3,0.911380,1.351361,1.349951,-0.212816,2.914717,3.236675,0.813772,0.101228,-0.526319,1.056650,1
4,0.173844,-0.537931,-0.697958,-0.631959,-0.627534,0.132669,-0.889648,-0.526551,-0.123391,-0.926871,1


### QDA

In [30]:
qda_model_best = QuadraticDiscriminantAnalysis(reg_param = 0.079248)
qda_model_best.fit(X, y)

labels_qda = pd.DataFrame(columns = ['ID', 'Label'])
labels_qda['Label'] = pd.DataFrame(qda_model_best.predict(test_data))
labels_qda['ID'] = labels_qda.index + 1
labels_qda

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [31]:
labels_qda.to_csv('new_predictions/qda_best.csv', index = False)

### Logistic Regression

In [32]:
logreg_model_best = LogisticRegression(C = best_C, random_state = 42)
logreg_model_best.fit(X, y)

labels_logreg = pd.DataFrame(columns = ['ID', 'Label'])
labels_logreg['Label'] = pd.DataFrame(logreg_model_best.predict(test_data))
labels_logreg['ID'] = labels_logreg.index + 1
labels_logreg

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
...,...,...
111,112,0
112,113,0
113,114,1
114,115,0


In [33]:
labels_logreg.to_csv('new_predictions/logreg_best.csv', index = False)

In [34]:
(labels_qda == labels_logreg).value_counts()

ID    Label
True  True     96
      False    20
Name: count, dtype: int64